<a href="https://colab.research.google.com/github/RasheedBlake/rb_gpt/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#version 2

import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
#import argparse
"""

parser = argparse.ArgumentParser(description='This is a demo program')
parser.add_argument('-batch_size', type=str, required=True, help="Please provide a batch_size")
args = parser.parse_args()
print(f'batch size: {args.batch_size}')
"""
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 64
block_size    = 256
max_iters     = 12000
learning_rate = 2.5e-4
eval_iters    = 500
n_embd        = 512
n_head        = 8
n_layer       = 12
dropout       = 0.10


chars = ""
with open("drive/MyDrive/Amazon Review LLM Data/amazon 1 train.csv", "r", encoding="utf-8") as f:
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])


# assumes globals exist: n_embd, n_head, n_layer, block_size, dropout, vocab_size, device, decode

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # causal mask sized to maximum context (block_size)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        # mask to current T; tril is already a buffer on the correct device
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),                    # fixed capitalization and enabled
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size  # remember maximum context length
        self.token_embedding_table   = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # clamp context to last block_size tokens to avoid pos-embed overflow
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)                       # (B,T,C)
        pos = torch.arange(T, device=index.device)                        # (T,)
        pos_emb = self.position_embedding_table(pos)                      # (T,C)
        x = tok_emb + pos_emb                                             # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                          # (B,T,vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            # flatten for CE
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        with torch.no_grad():
          for _ in range(max_new_tokens):
              idx_cond = idx[:, -block_size:]                 # use the growing idx
              logits, _ = self.forward(idx_cond)
              logits = logits[:, -1, :]                       # last step
              probs = F.softmax(logits, dim=-1)
              idx_next = torch.multinomial(probs, num_samples=1)
              idx = torch.cat((idx, idx_next), dim=1)
          return idx


# ---- usage ----
model = GPTLanguageModel(vocab_size)

print('loading model parameters...')
with open ('drive/MyDrive/Amazon Review LLM Data/model-02.pkl', 'rb') as f:
  model = pickle.load(f)
print('loaded successfully')

m = model.to(device)

while True:
    prompt = input("Prompt:\n")
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    generated_chars = decode(m.generate(context.unsqueeze(0), max_new_tokens=500)[0].tolist())
    print(f'Completion:\n{generated_chars}')

#context = torch.zeros((1, 1), dtype=torch.long, device=device)
#generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
#print(generated_chars)



cuda
loading model parameters...
loaded successfully
Prompt:
I just bought a new gaming controller from Amazon and I love it. It feels like good quality and the red and white design is really nice. Something I don't like about the controller is that one of the buttons feels a little stiff but other than that, it's good.
Completion:
I just bought a new gaming controller from Amazon and I love it. It feels like good quality and the red and white design is really nice. Something I don't like about the controller is that one of the buttons feels a little stiff but other than that, it's good. It doesn't protect the muscles, the steamers turn right in and strips out very easily and doesn't work. Once you put the tubes in the top, it ends up being made almost eopoly."
"2","HUGE SLOW MOVE","I picked this home-theater/slow move because it is house and slow movement the 44meg unit fails and doesn't have any problems at all. I can't keep homemidifying any of minor circular movements and it teache

KeyboardInterrupt: Interrupted by user

In [ ]:

#version 3
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------
# Device & hyperparams
# ------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 64
block_size    = 256
max_iters     = 12000
learning_rate = 2.5e-4
eval_iters    = 500
n_embd        = 512
n_head        = 8
n_layer       = 12
dropout       = 0.10

# ------------------
# Model components
# ------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # causal mask sized to maximum context (block_size)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        # mask to current T; tril is already a buffer on the correct device
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size  # remember maximum context length
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # clamp context to last block_size tokens to avoid pos-embed overflow
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)  # (B,T,C)
        pos = torch.arange(T, device=index.device)   # (T,)
        pos_emb = self.position_embedding_table(pos) # (T,C)
        x = tok_emb + pos_emb                        # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                     # (B,T,vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2 * T2, C), targets.view(B2 * T2))
        return logits, loss

    def generate(self, index, max_new_tokens):
        """Autoregressive sampling that conditions on newly generated tokens."""
        self.eval()
        idx = index
        with torch.no_grad():
            for _ in range(max_new_tokens):
                idx_cond = idx[:, -block_size:]            # use the growing sequence
                logits, _ = self.forward(idx_cond)
                logits = logits[:, -1, :]                  # (B, vocab) for last time step
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
                idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ------------------
# Load checkpoint (state_dict + vocab meta) and set up encode/decode
# ------------------
# Expecting a checkpoint saved like:
torch.save({"model_state": model.state_dict(),
"meta": {"stoi": string_to_int, "itos": int_to_string, "vocab_size": vocab_size}},
"model-02.pt")
ckpt_path = "drive/MyDrive/Amazon Review LLM Data/model-02.pt"
print("loading model checkpoint...")
ckpt = torch.load(ckpt_path, map_location=device)
meta = ckpt["meta"]
string_to_int = meta["stoi"]
int_to_string = meta["itos"]
vocab_size = meta["vocab_size"]

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda ids: ''.join(int_to_string[i] for i in ids)

model = GPTLanguageModel(vocab_size).to(device)
model.load_state_dict(ckpt["model_state"])
print("loaded successfully")

# ------------------
# Interactive loop
# ------------------
m = model  # alias for readability

while True:
    prompt = input("Prompt:\n")
    if prompt is None:
        break
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    out_idx = m.generate(context, max_new_tokens=150)[0].tolist()
    generated_text = decode(out_idx)
    print(f"Completion:\n{generated_text}")


cuda


NameError: name 'model' is not defined

In [ ]:
# version 4

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------
# Device & hyperparams
# ------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 12
#block_size    = 1024   # ↑ longer context; reduce batch if you hit OOM
max_iters     = 400_000
learning_rate = 2.5e-4
eval_iters    = 150           # faster evals (tune 100–200 as you like)
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.05

# ------------------
# Model components
# ------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # causal mask sized to maximum context (block_size)
        # self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # This will be initialized after loading meta
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        # mask to current T; tril is already a buffer on the correct device
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()
        self.block_size = block_size  # remember maximum context length
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

        # Initialize tril buffers in Head modules after block_size is known
        for block in self.blocks:
            for head in block.sa.heads:
                head.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # clamp context to last block_size tokens to avoid pos-embed overflow
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)  # (B,T,C)
        pos = torch.arange(T, device=index.device)   # (T,)
        pos_emb = self.position_embedding_table(pos) # (T,C)
        x = tok_emb + pos_emb                        # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                     # (B,T,vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.reshape(B2 * T2, C), targets.reshape(B2 * T2))
        return logits, loss

    def generate(self, index, max_new_tokens):
        """Autoregressive sampling that conditions on newly generated tokens."""
        self.eval()
        idx = index
        with torch.no_grad():
            for _ in range(max_new_tokens):
                idx_cond = idx[:, -self.block_size:]            # use the growing sequence
                logits, _ = self.forward(idx_cond)
                logits = logits[:, -1, :]                  # (B, vocab) for last time step
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
                idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ------------------
# Load checkpoint (state_dict + vocab meta) and set up encode/decode
# ------------------
ckpt_path = "drive/MyDrive/txt data/models/model-07.pt"
print("loading model checkpoint...")

ckpt = torch.load(ckpt_path, map_location=device)
if "meta" not in ckpt or "model_state" not in ckpt:
    raise RuntimeError(
        f"Checkpoint at {ckpt_path} is missing 'meta' or 'model_state'. "
        "Re-save your training checkpoint with both."
    )

meta = ckpt["meta"]
required_meta_keys = {"stoi", "itos", "vocab_size", "block_size"} # Added block_size to required keys
missing = required_meta_keys - set(meta.keys())
if missing:
    raise RuntimeError(f"Checkpoint meta missing keys: {missing}. Re-save with those fields.")

string_to_int = meta["stoi"]
int_to_string = meta["itos"]
vocab_size = meta["vocab_size"]
block_size = meta["block_size"] # Get block_size from meta

def encode(s: str):
    try:
        return [string_to_int[c] for c in s]
    except KeyError as e:
        bad = e.args[0]
        raise ValueError(
            f"Character {repr(bad)} not in saved vocabulary. "
            "Use the same tokenizer/char set as training."
        )

def decode(ids):
    return ''.join(int_to_string[i] for i in ids)

# Remove the "_orig_mod." prefix from the keys in the state dictionary
state_dict = ckpt["model_state"]
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("_orig_mod."):
        new_state_dict[k[len("_orig_mod."):]] = v
    else:
        new_state_dict[k] = v

model = GPTLanguageModel(vocab_size, block_size).to(device) # Pass block_size to the model
model.load_state_dict(new_state_dict)  # strict=True by default
print("loaded successfully")

# ------------------
# Interactive loop
# ------------------
m = model  # alias for readability
print("Type your prompt. Enter 'quit' to exit.")

while True:
    try:
        prompt = input("Prompt:\n")
    except (EOFError, KeyboardInterrupt):
        break
    if prompt is None or prompt.strip().lower() in {"quit", "exit"}:
        break
    if prompt.strip() == "":
        continue
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    out_idx = m.generate(context, max_new_tokens=150)[0].tolist() # Increased max_new_tokens for more output
    generated_text = decode(out_idx)
    print(f"Completion:\n{generated_text}\n")

cuda
loading model checkpoint...


RuntimeError: Error(s) in loading state_dict for GPTLanguageModel:
	Missing key(s) in state_dict: "lm_head.bias". 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

# ------------------
# Device
# ------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# ------------------
# Defaults (will be overwritten by ckpt meta)
# ------------------
n_embd  = 640
n_head  = 10
n_layer = 12
dropout = 0.05

# ------------------
# Model components (match training architecture)
# ------------------
class Head(nn.Module):
    def __init__(self, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, block_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_, block_size):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa   = MultiHeadAttention(n_head_, head_size, block_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1  = nn.LayerNorm(n_embd_)
        self.ln2  = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        # match training: bias=False, and tie weights
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding_table.weight  # tie

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# ------------------
# Load checkpoint (prefers EMA) and tokenizer meta
# ------------------
def load_checkpoint(path):
    # Try main path; fallback to "-final.pt" if exists
    try:
        return torch.load(path, map_location=device)
    except FileNotFoundError:
        base, ext = os.path.splitext(path)
        alt = base.replace('-final', '') + '-final' + ext
        print(f"Primary ckpt not found. Trying {alt} ...")
        return torch.load(alt, map_location=device)

ckpt_path = "drive/MyDrive/txt data/models/model-11.pt"  # adjust if needed
print(f"Loading {ckpt_path} ...")
ckpt = load_checkpoint(ckpt_path)

if "meta" not in ckpt:
    raise RuntimeError("Checkpoint missing 'meta'. Re-save training ckpt with meta.")
meta = ckpt["meta"]
for k in ("stoi","itos","vocab_size","block_size"):
    if k not in meta:
        raise RuntimeError(f"Checkpoint meta missing key: {k}")

# override architecture from meta to ensure shape-compat
vocab_size = int(meta["vocab_size"])
block_size = int(meta["block_size"])
n_embd  = int(meta.get("n_embd", n_embd))
n_head  = int(meta.get("n_head", n_head))
n_layer = int(meta.get("n_layer", n_layer))

stoi = meta["stoi"]; itos = meta["itos"]

def encode(s: str):
    # strict: raise if unseen char
    ids = []
    for c in s:
        if c not in stoi:
            raise ValueError(f"Character {repr(c)} not in saved vocabulary.")
        ids.append(stoi[c])
    return ids

def decode(ids):
    return ''.join(itos[i] for i in ids)

# prefer EMA weights if present
state = ckpt.get("ema_state", ckpt.get("model_state"))
if state is None:
    raise RuntimeError("Checkpoint missing 'model_state' (and no 'ema_state').")

# remove torch.compile prefix if present
clean_state = { (k.replace("_orig_mod.", "")): v for k, v in state.items() }

# build & load
model = GPTLanguageModel(vocab_size, block_size).to(device)
missing, unexpected = model.load_state_dict(clean_state, strict=False)
if missing:
    print("Warning: missing keys:", missing)
if unexpected:
    print("Warning: unexpected keys:", unexpected)
model.eval()
print("Loaded successfully (using EMA weights)" if "ema_state" in ckpt else "Loaded successfully (model weights)")

# ------------------
# Interactive loop
# ------------------
print("Type your prompt. Enter 'quit' to exit.")
while True:
    try:
        prompt = input("Prompt:\n")
    except (EOFError, KeyboardInterrupt):
        break
    if prompt is None or prompt.strip().lower() in {"quit", "exit"}:
        break
    if prompt.strip() == "":
        continue
    ctx = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(ctx, max_new_tokens=500)[0].tolist()
    print("Completion:\n" + decode(out) + "\n")


device: cuda
Loading drive/MyDrive/txt data/models/model-11.pt ...
Loaded successfully (using EMA weights)
Type your prompt. Enter 'quit' to exit.
Prompt:
I just bought a new gaming controller from Amazon and I love it. It feels like good quality and the red and white design is really nice. I one of the buttons feels a little stiff but other than that, it's good.
Completion:
I just bought a new gaming controller from Amazon and I love it. It feels like good quality and the red and white design is really nice. I one of the buttons feels a little stiff but other than that, it's good.

We have a new one in the design all around our home, we'll see it highlighting an interesting unexpected trick, and familiar topic. But Carter is making this list the top 100 people in this list so long as they're affected by our magic and what's happening. It's what we've made for the win, and the world is really supposed to be.

But three previous lists about this one  and they get a good deal of first-pe

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import textwrap

# ------------------
# Device
# ------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# ------------------
# Defaults (will be overwritten by ckpt meta)
# ------------------
n_embd  = 640
n_head  = 10
n_layer = 12
dropout = 0.05

# ------------------
# Model components (match training architecture)
# ------------------
class Head(nn.Module):
    def __init__(self, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, block_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_, block_size):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa   = MultiHeadAttention(n_head_, head_size, block_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1  = nn.LayerNorm(n_embd_)
        self.ln2  = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding_table.weight  # tie weights

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens, temperature=1.0):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# ------------------
# Load checkpoint (prefers EMA) and tokenizer meta
# ------------------
def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device)
    except FileNotFoundError:
        base, ext = os.path.splitext(path)
        alt = base.replace('-final', '') + '-final' + ext
        print(f"Primary ckpt not found. Trying {alt} ...")
        return torch.load(alt, map_location=device)

ckpt_path = "drive/MyDrive/txt data/models/model-11.pt"
print(f"Loading {ckpt_path} ...")
ckpt = load_checkpoint(ckpt_path)

if "meta" not in ckpt:
    raise RuntimeError("Checkpoint missing 'meta'. Re-save training ckpt with meta.")
meta = ckpt["meta"]
for k in ("stoi","itos","vocab_size","block_size"):
    if k not in meta:
        raise RuntimeError(f"Checkpoint meta missing key: {k}")

vocab_size = int(meta["vocab_size"])
block_size = int(meta["block_size"])
n_embd  = int(meta.get("n_embd", n_embd))
n_head  = int(meta.get("n_head", n_head))
n_layer = int(meta.get("n_layer", n_layer))

stoi = meta["stoi"]; itos = meta["itos"]

def encode(s: str):
    ids = []
    for c in s:
        if c not in stoi:
            raise ValueError(f"Character {repr(c)} not in saved vocabulary.")
        ids.append(stoi[c])
    return ids

def decode(ids):
    return ''.join(itos[i] for i in ids)

state = ckpt.get("ema_state", ckpt.get("model_state"))
if state is None:
    raise RuntimeError("Checkpoint missing 'model_state' (and no 'ema_state').")

clean_state = { (k.replace("_orig_mod.", "")): v for k, v in state.items() }

model = GPTLanguageModel(vocab_size, block_size).to(device)
missing, unexpected = model.load_state_dict(clean_state, strict=False)
if missing:
    print("Warning: missing keys:", missing)
if unexpected:
    print("Warning: unexpected keys:", unexpected)
model.eval()
print("Loaded successfully (using EMA weights)" if "ema_state" in ckpt else "Loaded successfully (model weights)")

# ------------------
# Interactive loop
# ------------------
try:
    temperature = float(input("Set temperature (e.g., 0.7, 1.0, 1.3): ") or "1.0")
except ValueError:
    temperature = 1.0
print(f"Using temperature = {temperature}")

wrapper = textwrap.TextWrapper(width=80)

def bubble(label, text, color='blue'):
    """Pretty prints text inside a pseudo chat bubble."""
    line = '─' * 78
    border_top = f"╭{line}╮"
    border_bottom = f"╰{line}╯"
    wrapped = '\n'.join(wrapper.wrap(text))
    print(f"\n{border_top}\n│ {label} │\n{border_bottom}")
    print(f"{wrapped}\n")

print("\nType your prompt. Enter 'quit' to exit.")
while True:
    try:
        prompt = input("\n🟢 Enter Prompt:\n")
    except (EOFError, KeyboardInterrupt):
        break
    if prompt is None or prompt.strip().lower() in {"quit", "exit"}:
        break
    if prompt.strip() == "":
        continue

    ctx = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(ctx, max_new_tokens=500, temperature=temperature)[0].tolist()
    completion = decode(out[len(encode(prompt)):])

    # Print "bubble" style output
    print("\n🔹" + "─" * 60)
    print("🟢 PROMPT:")
    print(wrapper.fill(prompt))
    print("\n🔵 COMPLETION:")
    print(wrapper.fill(completion))
    print("🔹" + "─" * 60 + "\n")


device: cuda
Loading drive/MyDrive/txt data/models/model-11.pt ...
Loaded successfully (using EMA weights)
Set temperature (e.g., 0.7, 1.0, 1.3): 0.7
Using temperature = 0.7

Type your prompt. Enter 'quit' to exit.

🟢 Enter Prompt:
I just bought a new gaming controller from Amazon and I love it. It feels like good quality and the red and white design is really nice. Something I don't like about the controller is that one of the buttons feels a little stiff but other than that, it's good.

🔹────────────────────────────────────────────────────────────
🟢 PROMPT:
I just bought a new gaming controller from Amazon and I love it. It feels like
good quality and the red and white design is really nice. Something I don't like
about the controller is that one of the buttons feels a little stiff but other
than that, it's good.

🔵 COMPLETION:
 The stuff is better and that's better to be done by the controller as well. The
controller is a brand new controller that connects to the previous stuff and 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import textwrap

# ------------------
# Device
# ------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# ------------------
# Defaults (will be overwritten by ckpt meta)
# ------------------
n_embd  = 640
n_head  = 10
n_layer = 12
dropout = 0.05

# ------------------
# Model components
# ------------------
class Head(nn.Module):
    def __init__(self, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, block_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_, block_size):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa   = MultiHeadAttention(n_head_, head_size, block_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1  = nn.LayerNorm(n_embd_)
        self.ln2  = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding_table.weight
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens, temperature=1.0, top_k=None, top_p=None,
                 repetition_penalty=1.0, ngram_block=3):
        """Generate tokens with temperature, top-k, top-p, repetition penalty, and n-gram blocking."""
        self.eval()
        idx = index
        seen_ngrams = set()  # store previously seen n-grams

        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :] / temperature

            # ----- repetition penalty -----
            if repetition_penalty != 1.0:
                for token_id in set(idx[0].tolist()):
                    logits[0, token_id] /= repetition_penalty

            probs = F.softmax(logits, dim=-1)

            # ----- top-k -----
            if top_k is not None and top_k > 0:
                values, _ = torch.topk(probs, top_k)
                min_values = values[:, -1].unsqueeze(1)
                probs = torch.where(probs < min_values, torch.zeros_like(probs), probs)
                probs = probs / probs.sum(dim=-1, keepdim=True)

            # ----- top-p (nucleus) -----
            if top_p is not None and 0.0 < top_p < 1.0:
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                mask = cumulative_probs > top_p
                mask[:, 1:] = mask[:, :-1].clone()
                mask[:, 0] = False
                sorted_probs[mask] = 0
                sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
                probs = torch.zeros_like(probs).scatter(1, sorted_indices, sorted_probs)

            # ----- sample next token -----
            next_token = torch.multinomial(probs, 1)

            # ----- n-gram blocking -----
            if ngram_block > 0 and idx.size(1) >= ngram_block - 1:
                prev_tokens = tuple(idx[0, -(ngram_block - 1):].tolist())
                candidate = prev_tokens + (next_token.item(),)
                tries = 0
                while candidate in seen_ngrams and tries < 10:
                    next_token = torch.multinomial(probs, 1)
                    candidate = prev_tokens + (next_token.item(),)
                    tries += 1
                seen_ngrams.add(candidate)

            idx = torch.cat([idx, next_token], dim=1)

        return idx

# ------------------
# Load checkpoint
# ------------------
def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device)
    except FileNotFoundError:
        base, ext = os.path.splitext(path)
        alt = base.replace('-final', '') + '-final' + ext
        print(f"Primary ckpt not found. Trying {alt} ...")
        return torch.load(alt, map_location=device)

ckpt_path = "drive/MyDrive/txt data/models/plaintext-wikipedia.pt"
print(f"Loading {ckpt_path} ...")
ckpt = load_checkpoint(ckpt_path)
meta = ckpt["meta"]
vocab_size = int(meta["vocab_size"])
block_size = int(meta["block_size"])
stoi = meta["stoi"]; itos = meta["itos"]
state = ckpt.get("ema_state", ckpt.get("model_state"))
clean_state = { (k.replace("_orig_mod.", "")): v for k, v in state.items() }

model = GPTLanguageModel(vocab_size, block_size).to(device)
model.load_state_dict(clean_state, strict=False)
model.eval()
print("Loaded successfully (using EMA weights)" if "ema_state" in ckpt else "Loaded successfully (model weights)")

# ------------------
# Interactive loop
# ------------------
try:
    temperature = float(input("Set temperature (e.g., 0.7, 1.0, 1.3): ") or "1.0")
except ValueError:
    temperature = 1.0
try:
    top_k = int(input("Set top_k (e.g., 0 for none, 50, 100): ") or "0")
except ValueError:
    top_k = 0
try:
    top_p = float(input("Set top_p (0.0–1.0, e.g., 0.9, 1.0 for none): ") or "1.0")
except ValueError:
    top_p = 1.0
try:
    repetition_penalty = float(input("Set repetition_penalty (1.0 for off, 1.1–1.3 typical): ") or "1.0")
except ValueError:
    repetition_penalty = 1.0

print(f"Using temperature={temperature}, top_k={top_k}, top_p={top_p}, repetition_penalty={repetition_penalty}")

wrapper = textwrap.TextWrapper(width=80)

def bubble(label, text):
    line = '─' * 78
    border_top = f"╭{line}╮"
    border_bottom = f"╰{line}╯"
    wrapped = '\n'.join(wrapper.wrap(text))
    print(f"\n{border_top}\n│ {label} │\n{border_bottom}")
    print(f"{wrapped}\n")

print("\nType your prompt. Enter 'quit' to exit.")
while True:
    try:
        prompt = input("\n🟢 Enter Prompt:\n")
    except (EOFError, KeyboardInterrupt):
        break
    if prompt is None or prompt.strip().lower() in {"quit", "exit"}:
        break
    if prompt.strip() == "":
        continue

    ctx = torch.tensor([stoi[c] for c in prompt if c in stoi], dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(ctx, max_new_tokens=500,
                         temperature=temperature,
                         top_k=top_k if top_k > 0 else None,
                         top_p=top_p if top_p < 1.0 else None,
                         repetition_penalty=repetition_penalty,
                         ngram_block=3)[0].tolist()

    completion = ''.join(itos[i] for i in out[len(ctx[0]):])

    print("\n🔹" + "─" * 60)
    print("🟢 PROMPT:")
    print(wrapper.fill(prompt))
    print("\n🔵 COMPLETION:")
    print(wrapper.fill(completion))
    print("🔹" + "─" * 60 + "\n")


device: cuda
Loading drive/MyDrive/txt data/models/plaintext-wikipedia.pt ...
Loaded successfully (using EMA weights)
Set temperature (e.g., 0.7, 1.0, 1.3): .8
Set top_k (e.g., 0 for none, 50, 100): 70
Set top_p (0.0–1.0, e.g., 0.9, 1.0 for none): .9
Set repetition_penalty (1.0 for off, 1.1–1.3 typical): 1.25
Using temperature=0.8, top_k=70, top_p=0.9, repetition_penalty=1.25

Type your prompt. Enter 'quit' to exit.

🟢 Enter Prompt:
The United States of America has 52 states. One of them being 

🔹────────────────────────────────────────────────────────────
🟢 PROMPT:
The United States of America has 52 states. One of them being

🔵 COMPLETION:
Alaska, the Northwest Territory is the support rate for civil liberties.
Section 86 of the United States Census Bureau's 1970 U.S. census reported: For
example, US 61-307 in Darlington, New Jersey, in the town of Hooper (Oklahoma),
at a meeting of six adjoining states, 14 U.S. states, 8 Canada, 9 Canada, 2
October 1994, and 1 March 2000, by which t

🟢 PROMPT:
I made me and my boyfriend some breakfast. Usually I would watch a TV Show on my
phone while eating but I wasn't able to for five minutes because the code used
to log into my profile was saying error. I also tried to reset the code but I
had to put the password in for the account and that wasn't working either. I
then turned my phone off and back on again. I remembered that I turned on safe
browsing and VPN which was interfering with the streaming platform.

🟢 PROMPT:
The weather is starting to get cold and I wish that I was back home in Jamaica.
Jamaica is a

🟢 PROMPT:
I moved to the United States in 2016 when I was 12. I wasn't very used the cold
weather because Jamaica is always sunny and it never snows there. Now it's
October and I would love to be in that sunny weather again.


🟢 PROMPT:
I just bought a new gaming controller from Amazon and I love it. It feels like
good quality and the red and white design is really nice. Something I don't like
about the controller is that one of the buttons feels a little stiff but other
than that, it's good.


🟢 PROMPT:
I went to a haunted house last night and it wasn't as bad as I thought it would
be. I went with my girlfriend, mom and my little sister. They were scared of the
monsters in the haunted house but they didn't frighten me.


🟢 PROMPT:
Write a story about a superhero who saved the city from a villain named
"Manboy."

__
Using temperature=0.67, top_k=50, top_p=0.82, repetition_penalty=1.3
